DATA CLEANING

Begin Our PySpark Session.

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

Define Our Schema:

In [3]:
from pyspark.sql.types import StructType, StructField, StringType

# Here, ensure they are all StringType() as is the original data
raw_schema = StructType([
    StructField("listing_id",          StringType(), True),
    StructField("suburb",              StringType(), True),
    StructField("postcode",            StringType(), True),
    StructField("weekly_rent",         StringType(), True),
    StructField("bond",                StringType(), True),
    StructField("available_date",      StringType(), True),
    StructField("date_listed",         StringType(), True),
    StructField("days_listed",         StringType(), True),
    StructField("bedrooms",            StringType(), True),
    StructField("bathrooms",           StringType(), True),
    StructField("carspaces",           StringType(), True),
    StructField("property_type",       StringType(), True),
    StructField("address",             StringType(), True),
    StructField("lat",                 StringType(), True),
    StructField("lon",                 StringType(), True),
    StructField("scraped_date",        StringType(), True),
    StructField("domain_page_id",      StringType(), True),
    StructField("property_id",         StringType(), True),
    StructField("photo_count",         StringType(), True),
    StructField("video_count",         StringType(), True),
    StructField("floorplans_count",    StringType(), True),
    StructField("virtual_tour",        StringType(), True),
    StructField("primary_type",        StringType(), True),
    StructField("secondary_type",      StringType(), True),
    StructField("agency",              StringType(), True),
    StructField("agency_id",           StringType(), True),
    StructField("agent_names",         StringType(), True),
    StructField("structured_features", StringType(), True),
    StructField("url",                 StringType(), True),
    StructField("land_area",           StringType(), True),
])


Read in our data using our schema. 
Note: We must allow multi line csv reading since there are multiple rows within our data where the "structured_features" column contains multiline descriptions. If we do not clarify this, our data will not be alligned properly when read in.

In [4]:
path = "../rea_data/landing/domain/Data/vic_rentals_all.csv"
df_landing = spark.read.csv(
    path,
    schema=raw_schema,
    header=True,
    quote='"',
    escape='"',
    multiLine=True
)
df_landing.printSchema()

root
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: string (nullable = true)
 |-- bond: string (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: string (nullable = true)
 |-- days_listed: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- carspaces: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- scraped_date: string (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: string (nullable = true)
 |-- video_count: string (nullable = true)
 |-- floorplans_count: string (nullable = true)
 |-- virtual_tour: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (null

Lets define a function to check null values within our data for use throughout the preprocessing procedure.

In [5]:
from functools import reduce

def null_check(df):
    has_null_condition = reduce(
        lambda acc, c: acc | F.col(c).isNull(),
        df.columns,
        F.lit(False)
    )

    # Count total rows, rows with atleast one null and count how many nulls occur per
    # data field.
    summary = df.agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.sum(has_null_condition.cast("int")).alias("rows_with_at_least_one_null"),
        *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in df.columns]
    ).collect()[0]
    
    total = summary["total_rows"]
    rows_with_at_least_one_null = summary["rows_with_at_least_one_null"]
    
    print(f"Total rows: {total:,}")
    print(f"Rows with at least one null value: {rows_with_at_least_one_null:,}")
    print(f"Percentage of rows with atleast one NULL: {round(100 * rows_with_at_least_one_null / total, 2)}%")
    print()
    for c in df.columns:
        print(f"{c:25s} {summary[f'null_{c}']:>12,}")


In [6]:
null_check(df_landing)

Total rows: 12,717
Rows with at least one null value: 12,717
Percentage of rows with atleast one NULL: 100.0%

listing_id                           0
suburb                               0
postcode                             0
weekly_rent                        279
bond                               782
available_date                     136
date_listed                          4
days_listed                          4
bedrooms                           125
bathrooms                           51
carspaces                        1,803
property_type                        0
address                            104
lat                                  4
lon                                  4
scraped_date                         0
domain_page_id                       1
property_id                     10,145
photo_count                          4
video_count                          4
floorplans_count                     4
virtual_tour                         4
primary_type                   

Now, lets explicitly type cast so we can tell when data was genuinely blank in the source rather than a silent parsing error resulting in missing data. Note: We will only do this for relevant columns and leave the rest as strings for now (They will be removed later).

In [8]:
df_clean = (
    df_landing
    # --- Numeric IDs: source stores these as floats ("21230.0"), 
    #     cast via double first, then to long. Use try_cast defensively. ---
    .withColumn("listing_id",     F.expr("try_cast(listing_id AS DOUBLE)").cast("long"))
    .withColumn("agency_id",      F.expr("try_cast(agency_id AS DOUBLE)").cast("long"))

    # property_id: alphanumeric code (e.g. "SD-2640-DB") — NOT numeric, leave as string
    # no cast applied — stays StringType from raw_schema

    # --- Money: clean decimal strings ("460.0", "1994.0") — direct cast is safe ---
    .withColumn("weekly_rent", F.expr("try_cast(weekly_rent AS DOUBLE)"))
    .withColumn("bond",        F.expr("try_cast(bond AS DOUBLE)"))

    # --- Dates ---
    .withColumn("date_listed",  F.to_date("date_listed", "yyyy-MM-dd"))
    .withColumn("scraped_date", F.to_timestamp("scraped_date", "yyyy-MM-dd HH:mm:ss"))
    # available_date left as string, per earlier decision

    # --- Simple numerics ---
    .withColumn("days_listed",      F.expr("try_cast(days_listed AS DOUBLE)"))
    .withColumn("bedrooms",         F.expr("try_cast(bedrooms AS DOUBLE)"))
    .withColumn("bathrooms",        F.expr("try_cast(bathrooms AS DOUBLE)"))
    .withColumn("carspaces",        F.expr("try_cast(carspaces AS DOUBLE)"))
    .withColumn("photo_count",      F.expr("try_cast(photo_count AS DOUBLE)").cast("integer"))
    .withColumn("video_count",      F.expr("try_cast(video_count AS DOUBLE)").cast("integer"))
    .withColumn("floorplans_count", F.expr("try_cast(floorplans_count AS DOUBLE)").cast("integer"))
    .withColumn("lat", F.expr("try_cast(lat AS DOUBLE)"))
    .withColumn("lon", F.expr("try_cast(lon AS DOUBLE)"))

    # --- Boolean ---
    .withColumn("virtual_tour", F.col("virtual_tour").cast("boolean"))

    # --- land_area: extract numeric part ---
    .withColumn("land_area_sqm",
        F.regexp_extract(F.col("land_area"), r"(\d+\.?\d*)", 1).cast("double"))
)

In [9]:
df_clean.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- bond: double (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- days_listed: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- scraped_date: timestamp (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: integer (nullable = true)
 |-- video_count: integer (nullable = true)
 |-- floorplans_count: integer (nullable = true)
 |-- virtual_tour: boolean (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (n

Once again, lets do a null audit to ensure we haven't accidently removed any missing data.

In [10]:
null_check(df_clean)

Total rows: 12,717
Rows with at least one null value: 12,717
Percentage of rows with atleast one NULL: 100.0%

listing_id                           0
suburb                               0
postcode                             0
weekly_rent                        279
bond                               782
available_date                     136
date_listed                          4
days_listed                          4
bedrooms                           125
bathrooms                           51
carspaces                        1,803
property_type                        0
address                            104
lat                                  4
lon                                  4
scraped_date                         0
domain_page_id                       1
property_id                     10,145
photo_count                          4
video_count                          4
floorplans_count                     4
virtual_tour                         4
primary_type                   

Lets also clean up our "structued_features" descriptions so they are all comma seperated properly and consistent with other rows.

In [11]:
row = df_clean.filter(F.col("listing_id") == "17747859")
row.show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 listing_id          | 17747859                                                                                                                                                                                                                                                                                                                                  
 suburb              | DOCKLANDS                                                                                                                                                                                                                                                                    

In [12]:
df_clean = df_clean.withColumn(
    "structured_features",
    F.regexp_replace(
        F.col("structured_features"),
        r"[\r\n]+-\s*",   # matches \r\n or \n followed by "- " (the bullet style)
        ", "              # replace with a plain comma separator
    )
)

In [13]:
row = df_clean.filter(F.col("listing_id") == "17747859")
row.show(truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 listing_id          | 17747859                                                                                                                                                                                                                                                                                                                  
 suburb              | DOCKLANDS                                                                                                                                                                                                                                                                                                    

We used a specific case ("listing_id=17747859") to illustrate our point, but must check that it has been fixed for every such row.

In [14]:
remaining_bullets = df_clean.filter(
    F.col("structured_features").rlike(r"[\r\n]-\s")
)
remaining_bullets.count()

0

In [ ]:
# NOTE FOR LATER: IF WE TURN STRUCTURED FEATURES INTO COMMA SEPERATED ARRAYS OF FEATURES, something like "BUILDING AMENITIES" 
# IS NOT A FEATURE SO MUST BE PREPROCESSED BEFORE WE DO A STEP LIKE THAT.